# 代码 Prompt

## 代码 Prompt（中文）- LedgerScope 1.1.0

以下是你之前提供的代码库上下文 Prompt 的更新版本，已整合 1.1.0 的变更：

---

# LedgerScope 代码库上下文 Prompt（完整版）- 1.1.0

## 一、项目概述

LedgerScope 是一个**通用财务建模框架**，核心抽象包括 Variable、Model、Auditor、Pipeline、Engine、Analysis、Visualization。框架与业务解耦，可复用于外贸、电商、SaaS、制造业等场景。

### 当前版本约束（1.x）
- 仅支持单一商品模型
- 订单来源假设为广告渠道（无复购、无分销）
- 转化率恒定
- 折旧与资本支出为占位（返回 0）
- 无订单交付时间滞后
- 固定汇率

### 版本规划
| 版本 | 定位 |
|:---|:---|
| 1.x | 稳定版本，仅 PATCH 修复 |
| 2.x | 功能扩展（升级包、多渠道、多商品、复购 LTV） |
| 3.x | 时间维度（交付周期、季节性） |
| 4.x | 高级模拟（动态汇率、融资成本） |

---

## 二、core/ 基类

### 2.1 Variable 基类

**位置**: `src/core/base_variable.py`

**构造规则**：

| 输入 | 处理 |
|:---|:---|
| min, exp, max 全提供 | 直接使用 |
| 仅 exp | min = max = exp |
| 仅 min + max | exp = (min + max) / 2 |
| 仅 max | min = 0, exp = max / 2 |
| 全空 | 全部 None（占位变量） |
| 仅 min | 语义不清晰，不允许 |

**核心方法**：

| 方法 | 说明 |
|:---|:---|
| `get_value(ValueType)` | 根据策略返回对应值 |
| `get_random_value()` | 返回 [min, max] 范围内的随机值 |
| `get_range_values(num)` | 返回等间距线性空间数组 |
| `set_value(value)` | 固定为常量（不推荐） |

**设计决策**：
- 不提供默认业务边界
- 实例不可变
- 变量名与代码解耦（使用 `variable_names` 常量）

---

### 2.2 Model 基类（1.1.0 更新）

**位置**: `src/core/base_model.py`

**核心变更（1.1.0）**：数据获取职责上移到基类，计算函数签名大幅简化。

| 版本 | 计算函数签名 | 变量解析 |
|:---|:---|:---|
| 1.0.0 | `def func(optional_variables: dict, **kwargs) -> dict` | 函数内 `kwargs.get(key, default)` |
| 1.1.0 | `def func(variables: dict) -> dict` | 基类统一处理 |

**核心属性**：

| 属性 | 类型 | 说明 |
|:---|:---|:---|
| `_required_variables` | `list[str]` | 必需变量名 |
| `_optional_variables` | `dict[str, float]` | 可选变量名及其默认值 |
| `_model_function` | `callable` | 核心计算函数，签名 `(variables: dict) -> dict` |
| `_output_names` | `list[str]` | 输出变量名列表 |

**核心方法**：

| 方法 | 说明 |
|:---|:---|
| `check_variables()` | 验证必需变量存在 |
| `_get_variable_value(name, is_optional)` | 私有方法，解析单个变量 |
| `prepare_calculation_context()` | 合并 required + optional 为统一字典 |
| `evaluate()` | 验证 → 准备 context → 调用函数 → 合并结果 |

**_model_function 签名规范（1.1.0）**：

```python
def calculate_example(variables: dict) -> dict:
    """
    Args:
        variables: 统一执行上下文，包含所有 required 和 optional 变量
    Returns:
        {output_name: value}
    """
    revenue = variables["Revenue"]
    cost = variables["Cost"]
    return {"Profit": revenue - cost}
```

**设计决策**：
- Model 不依赖 Variable 类
- 就地更新策略（调用方需自行 copy 保留原始状态）

---

### 2.3 Auditor 基类（1.1.0 更新）

**位置**: `src/core/base_auditor.py`

**继承关系**：`Auditor` 继承 `Model`

**差异**：
- `output_names` 返回空列表
- `evaluate()` 调用 `prepare_calculation_context()` 后执行验证，不更新字典

**验证函数规范（1.1.0）**：

```python
def check_xxx(variables: dict) -> None:
    """
    Args:
        variables: 统一执行上下文
    Raises:
        ValueError: 验证失败时抛出
    """
    value = variables["SomeVariable"]
    if not condition:
        raise ValueError("清晰的错误信息")
```

**设计决策**：
- Auditor 是 Model 特化
- 验证失败中断 Pipeline

---

## 三、variables/ 变量定义

### 文件结构

| 文件 | 内容 |
|:---|:---|
| `advertising.py` | AdvertisingBudget, GoogleSearchConversionRate, GoogleSearchCostPerClick |
| `costs.py` | Cost, AdvertisingCost, SetupCost |
| `deals.py` | Orders, CloseRate, UnitExw, UnitRetail, UnitsPerOrder |
| `expenses.py` | Expense, MonthlyExpense, RentExpense |
| `finance.py` | TaxRate, USDToRMB, TariffRate, PriceToEarningsRatio |

### 变量定义模板

```python
from src.config import variable_names
from src.core import Variable

class MyVariable(Variable):
    def __init__(self, min=None, exp=None, max=None):
        super().__init__(min, exp, max)
        self._name = variable_names.MY_VARIABLE
```

### 重要说明

- Variable 仅覆盖**自变量**（模型输入）
- 因变量（如 Revenue, NetIncome）由 Model 计算，无 Variable 子类
- 但某些变量（如 Orders）可在简化分析中作为自变量使用，因此仍可定义为 Variable

---

## 四、models/ 模型实现

### 模型分类与清单（代表性示例）

**广告漏斗**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `AdvertisingEfficiencyGoogleSearchModel` | `Leads = (Budget × Allocation) / (CPC × USDToRMB) × CVR` | LEADS |

**成本**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `CostOfGoodsSoldModel` | `COGS = UnitExw × Orders × UnitsPerOrder` | COGS |
| `TotalCostModel` | `TotalCost = COGS + AdvertisingCost + ShippingCost` | COST |

**交易**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `OrderModel` | `Orders = Leads × CloseRate` | ORDERS |
| `UnitFobModel` | `UnitFob = UnitRetail × (1 - DeductionRate)` | UNIT_FOB |

**收入与利润**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `RevenueModel` | `Revenue = UnitFob × Orders × UnitsPerOrder × USDToRMB` | REVENUE |
| `NetIncomeModel` | `NetIncome = (Revenue - Cost - Expense - Depreciation) × (1 - TaxRate)` | NET_INCOME |

**指标**：

| 模型 | 公式 | 输出 |
|:---|:---|:---|
| `MarketPriceModel` | `MarketPrice = (NetIncome × 12 × PE) / Months` | MARKET_PRICE |

### 模型实现模板（1.1.0）

```python
from src.config import variable_names
from src.core import Model

def calculate_xxx(variables: dict) -> dict:
    """计算函数"""
    required_var = variables[variable_names.REQUIRED_VAR]
    optional_var = variables.get(variable_names.OPTIONAL_VAR, default_val)
    
    if denominator == 0:
        return {variable_names.OUTPUT: 0.0}
    
    result = ...
    return {variable_names.OUTPUT: result}

class XxxModel(Model):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = calculate_xxx
        self._output_names = [variable_names.OUTPUT]
        self._required_variables = [variable_names.REQUIRED_VAR]
        self._optional_variables = {variable_names.OPTIONAL_VAR: 0.0}
```

---

## 五、auditors/ 审计器

### PriceArchitectureAuditor

**验证规则**：
1. `COGS_per_unit + Profit_per_unit == UnitFob`
2. `UnitFob + ShippingCost_per_unit + Tariff_per_unit + RetailMargin_per_unit == UnitRetail`

**容差设置**：`AUDIT_REL_TOL` (1e-3)，`AUDIT_ABS_TOL` (1e-2)

### Auditor 模板（1.1.0）

```python
from src.core import Auditor

def check_xxx(variables: dict) -> None:
    var = variables[variable_names.VAR]
    if not condition:
        raise ValueError(f"验证失败: {var}")

class XxxAuditor(Auditor):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = check_xxx
        self._required_variables = [variable_names.VAR]
```

---

## 六、pipelines/ 管道

### 构建方式

| 方式 | 示例 |
|:---|:---|
| 直接使用模型类 | `[ModelA(), ModelB()]` |
| 通过名称列表 | `PipelineComposer.build_pipeline_by_keys(["a", "b"])` |
| 基于预定义场景 | `PipelineComposer.build_named_scenario("scenario_name")` |
| 合并多个场景 | `PipelineComposer.build_merged_scenarios(["a", "b"])` |

### 预定义场景（代表性）

| 场景名称 | 用途 |
|:---|:---|
| `costs` | 成本分析 |
| `marketing_roi_analysis` | 营销 ROI |
| `complete_macro_metrics` | 完整宏观指标 |

### 拓扑顺序验证

**黄金法则**：一个变量一旦被作为输入消费，就不能在后续模型中重新计算。

---

## 七、engine/ 执行引擎

| 函数 | 说明 |
|:---|:---|
| `evaluate_chained_models(state, pipeline)` | 顺序执行模型 |
| `evaluate_expected_scenario(variables, pipeline)` | 所有变量取期望值 |
| `evaluate_stochastic_iteration(variables, shuffled, pipeline)` | 单次随机采样 |
| `evaluate_variable_scenario_sweep(variables, var, values, pipeline)` | 单变量扫描 |

---

## 八、analysis/ 分析框架

| 函数 | 用途 |
|:---|:---|
| `break_even_analysis` | 盈亏平衡 |
| `comparative_statics` | 三点敏感性 |
| `stochastic_contribution_analysis` | 贡献度饼图 |
| `run_monte_carlo` | 蒙特卡洛模拟 |
| `stochastic_bivariate_simulation` | 回归分析 |
| `run_two_way_sensitivity_analysis` | 双变量热力图 |

---

## 九、visualization/ 可视化

| 视图 | 对应分析 | 输出 |
|:---|:---|:---|
| `render_break_even_dashboard` | break_even_analysis | Styler 表格 |
| `render_comparative_statics_dashboard` | comparative_statics | Styler 表格 |
| `generate_contribution_pie_chart` | contribution_analysis | 饼图 |
| `generate_histogram_from_array` | monte_carlo | 直方图 |
| `generate_linear_regression_from_lists` | regression | 散点图 + 回归线 |
| `generate_heatmap_from_df` | two_way_sensitivity | 热力图 |

---

## 十、config/ 配置

### settings.py

| 参数 | 默认值 | 说明 |
|:---|:---|:---|
| `NUMS_IN_RANGE` | 50 | 扫描步数 |
| `SAMPLE_SIZE` | 100 | 蒙特卡洛默认迭代 |
| `AUDIT_REL_TOL` | 1e-3 | 审计相对容差 |
| `DEFAULT_CURRENCY_RATE` | 6.8 | USD→RMB |

### variable_names.py

50+ 常量，分类：Expenses、Investment、Deals、Finance、Ads、Costs、Revenue、Metrics、Price Architecture、Break even、Comparative statics

---

## 十一、utils/ 工具

| 函数 | 说明 |
|:---|:---|
| `get_missing_elements(provided, required)` | 返回缺失变量列表 |
| `check_variables_for_function(provided, required)` | 缺失则抛出 KeyError |
| `check_model_pipeline_topology_order(models)` | 验证拓扑顺序 |
| `fmt(val, d=0, s='', p=False)` | 数值格式化 |

---

## 十二、关键设计决策汇总

| 决策 | 原因 |
|:---|:---|
| Variable 无默认边界 | 避免隐性假设 |
| Variable 不可变 | 保持可复现性 |
| Model 不依赖 Variable | 可独立测试 |
| Model 计算函数签名简化（1.1.0） | 消除样板代码，单一职责 |
| Auditor 是 Model 特化 | 统一接口，无缝嵌入 |
| 拓扑顺序验证 | 防止变量覆盖冲突 |
| SetupCost 不计入 TotalCost | 作为投资项，用于 ROI |

---

**文档版本**：1.1.0
**创建日期**：2026-06-13
**适用代码版本**：LedgerScope 1.1.0



# Code Prompt (English) - LedgerScope 1.1.0

---

# LedgerScope Codebase Context Prompt (Full Version) - 1.1.0

## I. Project Overview

LedgerScope is a **general-purpose financial modeling framework** with core abstractions including Variable, Model, Auditor, Pipeline, Engine, Analysis, and Visualization. The framework is decoupled from business logic and reusable across scenarios such as international trade, e-commerce, SaaS, and manufacturing.

### Current Version Constraints (1.x)
- Single product model only
- Orders assumed from paid advertising channels (no repeat purchases, no distribution)
- Constant conversion rates
- Depreciation and CapEx are placeholders (return 0)
- No order delivery time lag
- Fixed exchange rate

### Version Roadmap

| Version | Positioning |
|:---|:---|
| 1.x | Stable version, PATCH-only fixes |
| 2.x | Feature expansion (upgrades, multi-channel, multi-product, LTV) |
| 3.x | Time dimension (delivery lag, seasonality) |
| 4.x | Advanced simulation (dynamic exchange rates, financing costs) |

---

## II. core/ Base Classes

### 2.1 Variable Base Class

**Location**: `src/core/base_variable.py`

**Construction Rules**:

| Input | Handling |
|:---|:---|
| min, exp, max all provided | Use as-is |
| exp only | min = max = exp |
| min + max only | exp = (min + max) / 2 |
| max only | min = 0, exp = max / 2 |
| All empty | All None (placeholder) |
| min only | Not allowed (ambiguous) |

**Core Methods**:

| Method | Description |
|:---|:---|
| `get_value(ValueType)` | Returns value based on strategy |
| `get_random_value()` | Returns random value in [min, max] |
| `get_range_values(num)` | Returns evenly spaced numpy array |
| `set_value(value)` | Fixes to constant (not recommended) |

**Design Decisions**:
- No default business boundaries
- Immutable instances
- Variable names decoupled via `variable_names` constants

---

### 2.2 Model Base Class (1.1.0 Update)

**Location**: `src/core/base_model.py`

**Key Change (1.1.0)**: Data retrieval responsibility moved from calculation functions to the base class.

| Version | Function Signature | Variable Resolution |
|:---|:---|:---|
| 1.0.0 | `def func(optional_variables: dict, **kwargs) -> dict` | Inside function via `kwargs.get(key, default)` |
| 1.1.0 | `def func(variables: dict) -> dict` | Handled by base class |

**Core Attributes**:

| Attribute | Type | Description |
|:---|:---|:---|
| `_required_variables` | `list[str]` | Required variable names |
| `_optional_variables` | `dict[str, float]` | Optional variables with defaults |
| `_model_function` | `callable` | Core calculation function, signature `(variables: dict) -> dict` |
| `_output_names` | `list[str]` | Output variable names |

**Core Methods**:

| Method | Description |
|:---|:---|
| `check_variables()` | Validates required variables exist |
| `_get_variable_value(name, is_optional)` | Private method resolving a single variable |
| `prepare_calculation_context()` | Merges required + optional into unified dict |
| `evaluate()` | Validate → prepare context → call function → merge results |

**_model_function Signature (1.1.0)**:

```python
def calculate_example(variables: dict) -> dict:
    """
    Args:
        variables: Unified execution context with all required and optional variables
    Returns:
        {output_name: value}
    """
    revenue = variables["Revenue"]
    cost = variables["Cost"]
    return {"Profit": revenue - cost}
```

**Design Decisions**:
- Model does not depend on Variable class
- In-place update strategy (caller must copy to preserve original state)

---

### 2.3 Auditor Base Class (1.1.0 Update)

**Location**: `src/core/base_auditor.py`

**Inheritance**: `Auditor` inherits from `Model`

**Differences**:
- `output_names` returns empty list
- `evaluate()` calls `prepare_calculation_context()` then executes validation, does NOT update the dictionary

**Validation Function Specification (1.1.0)**:

```python
def check_xxx(variables: dict) -> None:
    """
    Args:
        variables: Unified execution context
    Raises:
        ValueError: On validation failure
    """
    value = variables["SomeVariable"]
    if not condition:
        raise ValueError("Clear error message")
```

**Design Decisions**:
- Auditor is a Model specialization
- Validation failure halts Pipeline

---

## III. variables/ Variable Definitions

### File Structure

| File | Contents |
|:---|:---|
| `advertising.py` | AdvertisingBudget, GoogleSearchConversionRate, GoogleSearchCostPerClick |
| `costs.py` | Cost, AdvertisingCost, SetupCost |
| `deals.py` | Orders, CloseRate, UnitExw, UnitRetail, UnitsPerOrder |
| `expenses.py` | Expense, MonthlyExpense, RentExpense |
| `finance.py` | TaxRate, USDToRMB, TariffRate, PriceToEarningsRatio |

### Variable Definition Template

```python
from src.config import variable_names
from src.core import Variable

class MyVariable(Variable):
    def __init__(self, min=None, exp=None, max=None):
        super().__init__(min, exp, max)
        self._name = variable_names.MY_VARIABLE
```

### Important Note

- Variable only covers **independent variables** (model inputs)
- Dependent variables (e.g., Revenue, NetIncome) are calculated by Models and have no Variable subclass
- However, some variables (e.g., Orders) may serve as independent variables in simplified analyses, so they can still be defined as Variable

---

## IV. models/ Model Implementations

### Model Categories & List (Representative Examples)

**Advertising Funnel**:

| Model | Formula | Output |
|:---|:---|:---|
| `AdvertisingEfficiencyGoogleSearchModel` | `Leads = (Budget × Allocation) / (CPC × USDToRMB) × CVR` | LEADS |

**Cost**:

| Model | Formula | Output |
|:---|:---|:---|
| `CostOfGoodsSoldModel` | `COGS = UnitExw × Orders × UnitsPerOrder` | COGS |
| `TotalCostModel` | `TotalCost = COGS + AdvertisingCost + ShippingCost` | COST |

**Deal**:

| Model | Formula | Output |
|:---|:---|:---|
| `OrderModel` | `Orders = Leads × CloseRate` | ORDERS |
| `UnitFobModel` | `UnitFob = UnitRetail × (1 - DeductionRate)` | UNIT_FOB |

**Income & Profit**:

| Model | Formula | Output |
|:---|:---|:---|
| `RevenueModel` | `Revenue = UnitFob × Orders × UnitsPerOrder × USDToRMB` | REVENUE |
| `NetIncomeModel` | `NetIncome = (Revenue - Cost - Expense - Depreciation) × (1 - TaxRate)` | NET_INCOME |

**Metrics**:

| Model | Formula | Output |
|:---|:---|:---|
| `MarketPriceModel` | `MarketPrice = (NetIncome × 12 × PE) / Months` | MARKET_PRICE |

### Model Implementation Template (1.1.0)

```python
from src.config import variable_names
from src.core import Model

def calculate_xxx(variables: dict) -> dict:
    """Calculation function"""
    required_var = variables[variable_names.REQUIRED_VAR]
    optional_var = variables.get(variable_names.OPTIONAL_VAR, default_val)
    
    if denominator == 0:
        return {variable_names.OUTPUT: 0.0}
    
    result = ...
    return {variable_names.OUTPUT: result}

class XxxModel(Model):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = calculate_xxx
        self._output_names = [variable_names.OUTPUT]
        self._required_variables = [variable_names.REQUIRED_VAR]
        self._optional_variables = {variable_names.OPTIONAL_VAR: 0.0}
```

---

## V. auditors/ Auditors

### PriceArchitectureAuditor

**Validation Rules**:
1. `COGS_per_unit + Profit_per_unit == UnitFob`
2. `UnitFob + ShippingCost_per_unit + Tariff_per_unit + RetailMargin_per_unit == UnitRetail`

**Tolerance Settings**: `AUDIT_REL_TOL` (1e-3), `AUDIT_ABS_TOL` (1e-2)

### Auditor Template (1.1.0)

```python
from src.core import Auditor

def check_xxx(variables: dict) -> None:
    var = variables[variable_names.VAR]
    if not condition:
        raise ValueError(f"Validation failed: {var}")

class XxxAuditor(Auditor):
    def __init__(self, input_variables: dict = None):
        super().__init__(input_variables)
        self._model_function = check_xxx
        self._required_variables = [variable_names.VAR]
```

---

## VI. pipelines/ Pipeline

### Construction Methods

| Method | Example |
|:---|:---|
| Direct model class list | `[ModelA(), ModelB()]` |
| Via model name list | `PipelineComposer.build_pipeline_by_keys(["a", "b"])` |
| Via predefined scenario | `PipelineComposer.build_named_scenario("scenario_name")` |
| Merge multiple scenarios | `PipelineComposer.build_merged_scenarios(["a", "b"])` |

### Predefined Scenarios (Representative)

| Scenario Name | Purpose |
|:---|:---|
| `costs` | Cost analysis |
| `marketing_roi_analysis` | Marketing ROI |
| `complete_macro_metrics` | Complete macro metrics |

### Topological Order Validation

**Golden Rule**: Once a variable has been consumed as input, it cannot be recomputed by downstream models.

---

## VII. engine/ Execution Engine

| Function | Description |
|:---|:---|
| `evaluate_chained_models(state, pipeline)` | Sequentially executes models |
| `evaluate_expected_scenario(variables, pipeline)` | All variables take expected values |
| `evaluate_stochastic_iteration(variables, shuffled, pipeline)` | Single random sampling |
| `evaluate_variable_scenario_sweep(variables, var, values, pipeline)` | Single variable sweep |

---

## VIII. analysis/ Analysis Framework

| Function | Purpose |
|:---|:---|
| `break_even_analysis` | Break-even analysis |
| `comparative_statics` | Three-point sensitivity |
| `stochastic_contribution_analysis` | Contribution pie chart |
| `run_monte_carlo` | Monte Carlo simulation |
| `stochastic_bivariate_simulation` | Regression analysis |
| `run_two_way_sensitivity_analysis` | Two-variable heatmap |

---

## IX. visualization/ Visualization

| View | Corresponding Analysis | Output |
|:---|:---|:---|
| `render_break_even_dashboard` | break_even_analysis | Styler table |
| `render_comparative_statics_dashboard` | comparative_statics | Styler table |
| `generate_contribution_pie_chart` | contribution_analysis | Pie chart |
| `generate_histogram_from_array` | monte_carlo | Histogram |
| `generate_linear_regression_from_lists` | regression | Scatter + regression line |
| `generate_heatmap_from_df` | two_way_sensitivity | Heatmap |

---

## X. config/ Configuration

### settings.py

| Parameter | Default | Description |
|:---|:---|:---|
| `NUMS_IN_RANGE` | 50 | Sweep steps |
| `SAMPLE_SIZE` | 100 | Default Monte Carlo iterations |
| `AUDIT_REL_TOL` | 1e-3 | Audit relative tolerance |
| `DEFAULT_CURRENCY_RATE` | 6.8 | USD→RMB |

### variable_names.py

50+ constants organized in categories: Expenses, Investment, Deals, Finance, Ads, Costs, Revenue, Metrics, Price Architecture, Break even, Comparative statics

---

## XI. utils/ Utilities

| Function | Description |
|:---|:---|
| `get_missing_elements(provided, required)` | Returns list of missing variables |
| `check_variables_for_function(provided, required)` | Raises KeyError if missing |
| `check_model_pipeline_topology_order(models)` | Validates topological order |
| `fmt(val, d=0, s='', p=False)` | Formats numeric values |

---

## XII. Key Design Decisions Summary

| Decision | Rationale |
|:---|:---|
| Variable has no default boundaries | Avoid implicit assumptions |
| Variable is immutable | Preserve reproducibility |
| Model does not depend on Variable | Enables independent testing |
| Model function signature simplified (1.1.0) | Eliminates boilerplate, single responsibility |
| Auditor is a Model specialization | Unified interface, seamless embedding |
| Topological order validation | Prevents variable overwrite conflicts |
| SetupCost excluded from TotalCost | Treated as investment item for ROI |

---

**Document Version**: 1.1.0
**Creation Date**: 2026-01-15
**Applicable Code Version**: LedgerScope 1.1.0

---



# LedgerScope 设计文档


## LedgerScope 设计文档上下文 Prompt

## 一、文档概述

LedgerScope 设计文档是一份完整的技术说明文档，面向两类读者：

| 读者类型 | 阅读目标 | 重点关注章节 |
|:---|:---|:---|
| **最终用户** | 了解框架能力、快速上手、查阅组件用法 | 第一部分（第 1-4 章）、第二部分（第 5-13 章） |
| **扩展开发者** | 学习如何添加新组件 | 第三部分（第 14-19 章）、附录 |

### 文档特性

- **自动目录**：使用 `[TOC]` 实现 GitHub 自动生成目录
- **可执行示例**：所有代码示例可直接复制运行（需安装依赖，详见附录 B.6）
- **Mermaid 图表**：类图可视化依赖关系
- **表格呈现**：变量清单、模型清单使用表格

### 文档结构

```
# LedgerScope Design Document

[TOC]

## Part 1: Overview & Getting Started (Chapters 1-4)
## Part 2: Component Reference Manual (Chapters 5-13)
## Part 3: Extension Development Guide (Chapters 14-19)
## Part 4: Appendices (A-G)
```

## 二、核心版本信息

### 当前版本：1.1.0（代码）/ 1.0（设计文档）

设计文档版本与代码版本解耦，设计文档保持宏观结构稳定。

**关键变更（1.1.0）**：Model 和 Auditor 的数据获取职责上移到基类。

| 组件 | 1.0.0 签名 | 1.1.0 签名 |
|:---|:---|:---|
| Model 计算函数 | `def func(optional_variables: dict, **kwargs) -> dict` | `def func(variables: dict) -> dict` |
| Auditor 验证函数 | `def check(optional_variables: dict, **kwargs) -> None` | `def check(variables: dict) -> None` |

### 版本规划

| 版本 | 定位 |
|:---|:---|
| 1.x | 稳定版本，仅 PATCH 修复 |
| 2.x | 功能扩展（升级包、多渠道、多商品、复购 LTV） |
| 3.x | 时间维度（交付周期、季节性） |
| 4.x | 高级模拟（动态汇率、融资成本） |

## 三、章节结构总览

### 第一部分：概览与入门（第 1-4 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 1 | 项目介绍 | 定位、设计原则、版本约束、未来规划 |
| 2 | 快速开始 | 环境安装、5 分钟示例、6 个分析案例速览 |
| 3 | 核心概念 | Variable、Model、Auditor、Pipeline、Engine、Analysis、Visualization |
| 4 | 使用流程 | 标准工作流、数据流向图、Jupyter 集成 |

### 第二部分：组件参考手册（第 5-13 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 5 | Variable 参考 | 变量分类、清单、设计决策 |
| 6 | Model 参考 | 模型分类、清单、依赖关系、设计决策 |
| 7 | Auditor 参考 | 审计器清单、设计决策 |
| 8 | Pipeline 参考 | 预定义场景、自定义管道、拓扑验证 |
| 9 | Analysis 参考 | 6 种分析模式、函数签名、使用示例 |
| 10 | Visualization 参考 | 架构设计、视图函数、格式化工具 |
| 11 | Config 参考 | settings、variable_names、messages、formatting、pipelines |
| 12 | Utils 参考 | validation、formatting、logger |
| 13 | 模块关系与依赖 | 架构模式、数据流、执行时序图 |

### 第三部分：扩展开发指南（第 14-19 章）

| 章节 | 标题 | 主要内容 |
|:---|:---|:---|
| 14 | 添加新 Variable | 步骤、命名规范、格式化配置 |
| 15 | 添加新 Model | 步骤、实现规范、计算函数（1.1.0 签名） |
| 16 | 添加新 Auditor | 步骤、实现规范、验证函数（1.1.0 签名） |
| 17 | 添加新 Pipeline | 步骤、场景配置、动态构建 |
| 18 | 添加新 Analysis | 步骤、函数模板、设计原则 |
| 19 | 添加新 Visualization | 步骤、视图模板、核心原则 |

### 第四部分：附录（A-G 章）

| 附录 | 标题 | 主要内容 |
|:---|:---|:---|
| A | 模型依赖图 | 核心组件类图（Mermaid） |
| B | 配置说明 | 系统参数、审计容差、场景配置、格式化映射、日志配置、外部依赖 |
| C | 完整示例 | 6 种分析模式的完整代码 |
| D | 已知局限与未来规划 | 版本约束、模型假设、Roadmap |
| E | 常见问题 | 调试技巧、性能优化、常见错误、模型设计建议 |
| F | 版本历史 | 版本概览、版本号规则（详见 CHANGELOG.md） |
| G | 作者信息 | 维护者、许可证、引用说明、贡献指南 |

## 四、关键设计决策

| 决策 | 说明 |
|:---|:---|
| Variable 无默认边界 | 避免隐性假设 |
| Variable 不可变 | 保持可复现性 |
| Model 不依赖 Variable | 可独立测试 |
| Model 计算函数签名简化（1.1.0） | 消除样板代码，单一职责 |
| Auditor 是 Model 特化 | 统一接口，无缝嵌入 |
| 拓扑顺序验证 | 防止变量覆盖冲突 |
| 样式与视图分离 | 便于主题定制 |

## 五、外部依赖

| 依赖库 | 用途 |
|:---|:---|
| `numpy` | 数组运算、线性空间生成 |
| `pandas` | 数据分析、DataFrame 操作 |
| `matplotlib` | 图表绘制 |
| `seaborn` | 热力图生成 |
| `statsmodels` | OLS 线性回归统计 |

## 六、格式规范

- 标题层级：`#` → `##` → `###` → `####`
- 表格：使用 `|:---|:---|:---|` 语法
- 代码块：````python```` 或 ````mermaid````
- 提示框：`> 💡 **提示**：` 或 `> ⚠️ **注意**：`
- 交叉引用：使用 `**第 X 章：章节名称**` 格式

---


# LedgerScope Design Document Context Prompt

## I. Document Overview

The LedgerScope Design Document is a comprehensive technical documentation aimed at two types of readers:

| Reader Type | Reading Goal | Key Sections |
|:---|:---|:---|
| **End Users** | Understand framework capabilities, quick start, component usage | Part 1 (Chapters 1-4), Part 2 (Chapters 5-13) |
| **Extension Developers** | Learn how to add new components | Part 3 (Chapters 14-19), Appendices |

### Document Features

- **Auto-generated table of contents**: Uses `[TOC]` for GitHub automatic generation
- **Executable examples**: All code examples can be copied and run directly (dependencies required, see Appendix B.6)
- **Mermaid diagrams**: Class diagrams visualize dependencies
- **Tabular presentation**: Variable and model lists use tables

### Document Structure

```
# LedgerScope Design Document

[TOC]

## Part 1: Overview & Getting Started (Chapters 1-4)
## Part 2: Component Reference Manual (Chapters 5-13)
## Part 3: Extension Development Guide (Chapters 14-19)
## Part 4: Appendices (A-G)
```

## II. Core Version Information

### Current Version: 1.1.0 (Code) / 1.0 (Design Document)

The design document version is decoupled from the code version to maintain macro-structural stability.

**Key Changes (1.1.0)**: Data retrieval responsibility moved to base classes for Model and Auditor.

| Component | 1.0.0 Signature | 1.1.0 Signature |
|:---|:---|:---|
| Model calculation function | `def func(optional_variables: dict, **kwargs) -> dict` | `def func(variables: dict) -> dict` |
| Auditor validation function | `def check(optional_variables: dict, **kwargs) -> None` | `def check(variables: dict) -> None` |

### Version Roadmap

| Version | Positioning |
|:---|:---|
| 1.x | Stable version, PATCH-only fixes |
| 2.x | Feature expansion (upgrade packages, multi-channel, multi-product, repeat purchase LTV) |
| 3.x | Time dimension (delivery lead time, seasonality) |
| 4.x | Advanced simulation (dynamic exchange rates, financing costs) |

## III. Chapter Structure Overview

### Part 1: Overview & Getting Started (Chapters 1-4)

| Chapter | Title | Main Content |
|:---|:---|:---|
| 1 | Project Introduction | Positioning, design principles, version constraints, roadmap |
| 2 | Quick Start | Environment setup, 5-minute example, 6 analysis examples at a glance |
| 3 | Core Concepts | Variable, Model, Auditor, Pipeline, Engine, Analysis, Visualization |
| 4 | Usage Workflow | Standard workflow, data flow diagram, Jupyter integration |

### Part 2: Component Reference Manual (Chapters 5-13)

| Chapter | Title | Main Content |
|:---|:---|:---|
| 5 | Variable Reference | Variable classification, list, design decisions |
| 6 | Model Reference | Model classification, list, dependencies, design decisions |
| 7 | Auditor Reference | Auditor list, design decisions |
| 8 | Pipeline Reference | Predefined scenarios, custom pipelines, topology validation |
| 9 | Analysis Reference | 6 analysis modes, function signatures, usage examples |
| 10 | Visualization Reference | Architecture design, view functions, formatting utilities |
| 11 | Config Reference | settings, variable_names, messages, formatting, pipelines |
| 12 | Utils Reference | validation, formatting, logger |
| 13 | Module Relationships & Dependencies | Architecture pattern, data flow, execution sequence diagram |

### Part 3: Extension Development Guide (Chapters 14-19)

| Chapter | Title | Main Content |
|:---|:---|:---|
| 14 | Adding New Variable | Steps, naming conventions, formatting configuration |
| 15 | Adding New Model | Steps, implementation specifications, calculation function (1.1.0 signature) |
| 16 | Adding New Auditor | Steps, implementation specifications, validation function (1.1.0 signature) |
| 17 | Adding New Pipeline | Steps, scenario configuration, dynamic construction |
| 18 | Adding New Analysis | Steps, function template, design principles |
| 19 | Adding New Visualization | Steps, view template, core principles |

### Part 4: Appendices (A-G)

| Appendix | Title | Main Content |
|:---|:---|:---|
| A | Model Dependency Diagram | Core component class diagram (Mermaid) |
| B | Configuration Reference | System parameters, audit tolerances, scenario configuration, formatting mapping, log configuration, external dependencies |
| C | Complete Examples | Complete code for 6 analysis modes |
| D | Known Limitations & Future Roadmap | Version constraints, model assumptions, roadmap |
| E | Frequently Asked Questions | Debugging tips, performance optimization, common errors, model design recommendations |
| F | Version History | Version overview, version numbering rules (see CHANGELOG.md) |
| G | Author Information | Maintainer, license, citation, contribution guidelines |

## IV. Key Design Decisions

| Decision | Rationale |
|:---|:---|
| Variable has no default boundaries | Avoid implicit assumptions |
| Variable is immutable | Maintain reproducibility |
| Model does not depend on Variable | Enables independent testing |
| Model calculation function signature simplified (1.1.0) | Eliminates boilerplate, single responsibility |
| Auditor is a Model specialization | Unified interface, seamless embedding |
| Topological order validation | Prevents variable overwrite conflicts |
| Separation of style and view | Facilitates theme customization |

## V. External Dependencies

| Library | Purpose |
|:---|:---|
| `numpy` | Array operations, linear space generation |
| `pandas` | Data analysis, DataFrame operations |
| `matplotlib` | Chart rendering |
| `seaborn` | Heatmap generation |
| `statsmodels` | OLS linear regression statistics |

## VI. Formatting Conventions

- Heading levels: `#` → `##` → `###` → `####`
- Tables: Use `|:---|:---|:---|` syntax
- Code blocks: ````python```` or ````mermaid````
- Callouts: `> 💡 **Tip**:` or `> ⚠️ **Note**:`
- Cross-references: Use `**Chapter X: Chapter Title**` format

---

## 调研报告-第一次校正版

# 泡泡屋财务调研报告 — 项目上下文与启动 Prompt


## 一、项目身份

您正在参与一份关于**泡泡屋（PC透明星空房）北美B2B出口业务**的财务调研报告的修订工作。该报告基于 LedgerScope 财务模型系统，采用"三阶段（实验期/优化期/稳定期）× 三情景（保守/基准/乐观）"的二维设计，通过蒙特卡洛模拟、敏感性分析、回归分析等工具完成全部计算。

**报告当前状态**：已完成全文撰写与整体性修订，包含执行摘要、16个正文章节及附录。报告采用 Mermaid 流程图、Markdown 表格、代码块等格式呈现。


## 二、报告结构总览

| 部分 | 涵盖章节 | 核心内容 |
|:---|:---|:---|
| **前置** | 执行摘要 | 一句话结论、核心数据速览、参数影响力排序、风险提示、最终建议 |
| **第一部分：调研基础** | 第1-2章 | 项目背景、调研目的、方法论框架、参数定义与行业基准 |
| **第二部分：核心财务表现** | 第3-9章 | 指标定义、规模/利润/效率/平衡分析、三情景预测、成本结构 |
| **第三部分：深度分析** | 第10-14章 | 敏感性分析、盈亏平衡、热力图、蒙特卡洛、回归分析 |
| **第四部分：结论与模型边界** | 第15-16章 | 结论、行动建议、模型局限性、后续升级 |
| **附录** | 附录A-E | 完整参数表、假设清单、核心代码片段、局限性详细说明、术语表 |


## 三、报告核心结论（快速参考）

| 指标 | 数值 |
|:---|:---|
| 稳定期月净利润 | ¥55,255 |
| 稳定期净利润率 | 58.0% |
| 盈亏平衡时间 | 第1个月 |
| 回本周期 | 第3个月 |
| 6个月累计净利润（中位）| ¥150,359 |
| 6个月ROI（启动成本¥6,000-15,000）| 10.0-25.1倍 |
| 达标概率（≥¥50,000/月）| 56.52% |


## 四、关键参数速查

| 参数 | 保守（min）| 基准（exp）| 乐观（max）| 备注 |
|:---|:---|:---|:---|:---|
| 月广告预算 | ¥2,000 | ¥3,000 | ¥5,000 | 全阶段 |
| CPC（稳定期）| \$1.80 | \$2.20 | \$3.00 | 月5-6 |
| CVR（稳定期）| 4.0% | 5.5% | 7.5% | 月5-6 |
| Close Rate（稳定期）| 15% | 22% | 30% | 月5-6 |
| Unit EXW Price | ¥7,000 | ¥5,000 | ¥3,000 | — |
| Unit FOB Price | \$3,150 | \$4,028 | \$4,905 | Monte Carlo推导 |
| 启动成本 | ¥15,000 | ¥10,000 | ¥6,000 | 一次性 |
| 月度运营费用 | ¥5,000 | ¥4,000 | ¥3,000 | 租金+差旅+服务费 |


## 五、报告核心原则

### 5.1 术语规范

| 原术语（避免使用）| 应使用 |
|:---|:---|
| 悲观情景 | 保守参数组合 |
| 基准情景 | 基准预期 |
| 乐观情景 | 乐观参数组合 |
| 精益/标准/稳健启动 | 启动成本¥6,000/¥10,000/¥15,000 |
| 广告预算不足 | 广告支出占总成本X%，存在预算提升空间 |

### 5.2 措辞规范

- 避免危机式措辞（"最需要守护的底线"）
- 避免绝对化表述（"无需纠结"、"首要任务"）
- 避免超出数据支撑范围的强因果推论
- 用"建议优先行动"替代"首要任务"
- 用"安全边际最小，需重点关注"替代"最需要守护的底线"

### 5.3 结构定位

| 内容类型 | 应放置位置 |
|:---|:---|
| 方法论详细说明 | 第4章（规模类指标分析）|
| 方法论简略提及 | 第5-7章开头 |
| 术语定义 | 第3章（首次出现时定义）|
| 假设清单（初始版）| 第1章（6项）|
| 假设清单（完整版）| 第16章（9项）|
| 行业基准数据 | 第2章（呈现）、第15章（引用）|
| 行动建议 | 第15章 |
| 模型边界/局限性 | 第16章 |


## 六、常用分析工具与对应章节

| 分析工具 | 用途 | 章节 |
|:---|:---|:---|
| 成本结构贡献度分析 | 拆解各成本项占总成本比重 | 第9章 |
| 单变量敏感性分析 | 测算各参数净利润弹性 | 第10章 |
| 多变量盈亏平衡分析 | 寻找实现利润目标的参数组合 | 第11章 |
| 双变量热力图分析 | 可视化两组参数联合影响 | 第12章 |
| 蒙特卡洛模拟 | 评估净利润概率分布与达标概率 | 第13章 |
| 线性回归分析 | 测算各参数对净利润的解释力（R²）| 第14章 |


## 七、已知待处理事项

| 事项 | 状态 | 说明 |
|:---|:---|:---|
| FOB价格推导与独立变量分析的冲突 | 暂缓处理 | 待模型数据更新后再处理 |
| 第8章P25-P75区间改用独立叠加法 | 已修正 | 需确认数值已同步更新 |
| 附录A列头术语 | 已确认 | 保持"保守/基准/乐观"，min/exp/max为取值端点 |


## 八、常见修订任务类型

### 类型A：数值更新
当参数取值、行业基准或模拟结果需要更新时，需同步检查：
- 第2章参数表
- 第4-8章数据总览与趋势分析
- 第11章盈亏平衡阈值与安全边际
- 第13章蒙特卡洛均值与达标概率
- 第15章结论与行动建议
- 执行摘要

### 类型B：措辞修订
全文措辞应符合以下标准：
- 以数据为依据，避免过度推断
- 区分"弹性"（因果边际贡献）和"R²"（统计解释力）
- 行动建议以"建议优先"而非"必须"的语态呈现

### 类型C：结构调优
若需调整章节结构：
- 方法论说明统一在第4章详细展开
- 假设清单：第1章6项初始版 + 第16章9项完整版
- 行业基准：第2章呈现数据 + 第15章引用结论
- 行动建议集中在第15章
- 模型边界集中在第16章

### 类型D：图表更新
- A图（模型计算逻辑总览）位于第1章1.3节
- C图（决策流程图）位于第1章1.3节末尾
- 所有代码块占位符统一为 `## 代码块`


## 九、文件清单

本报告包含以下独立文件/内容块，可根据需要单独修改：

| 序号 | 内容 | 格式 |
|:---|:---|:---|
| 1 | 执行摘要 | Markdown |
| 2 | 第1章：调研背景与方法论 | Markdown + Mermaid |
| 3 | 第2章：参数定义与行业基准 | Markdown + 表格 |
| 4 | 第3章：核心财务指标定义 | Markdown + 表格 |
| 5 | 第4-9章：核心财务表现 | Markdown + 表格 |
| 6 | 第10-14章：深度分析 | Markdown + 表格 |
| 7 | 第15章：结论与行动建议 | Markdown |
| 8 | 第16章：模型局限性与后续升级 | Markdown |
| 9 | 附录A-E | Markdown + 表格 + 代码块 |


## 十、启动指令模板

如需在新对话中继续工作，请直接使用以下指令：

---

**启动指令**：

> 我正在继续一份关于泡泡屋北美B2B出口业务的财务调研报告的修订工作。报告已完成全文撰写与整体性修订，包含执行摘要、16个正文章节及附录。
>
> 我需要：[请在此处描述具体任务，如"更新第2章的CPC参数并同步修改第6-8章的相关数据" / "弱化第11章中关于Unit FOB Price的措辞" / "检查全文中'悲观/基准/乐观'术语是否已全部替换" / "在附录中补充XX参数的计算逻辑" / "请输出完整的第X章修订版" / "请检查第X章与第Y章之间的数值一致性"]
>
> 报告核心原则请参考项目上下文中的术语规范、措辞规范和结构定位。请逐章输出修改结果，每章确认后再继续。

---

**结束指令**：

> 本次修订到此结束。感谢您的协助。如有后续修改需求，我会在新对话中开启任务。

---